# Deepfake Sampling & Visualization - Robust Version (v2)
This notebook generates side-by-side visualizations of **Real Source Images** and their **Generated Deepfake** counterparts.

### v2 Fixes:
1.  **Robust Filename Matching:** Handles inconsistent 'f' prefixes (e.g., `f33692` vs `33692`) and case sensitivity.
2.  **Placeholder Support:** Displays a descriptive message if a source image is truly missing from the dataset instead of a blank plot.

In [ ]:
import os
import random
import zipfile
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive
from tqdm.auto import tqdm

## 1. Setup & Path Configuration

In [ ]:
BASE_PATH = '/content'
DRIVE_MOUNT = '/content/drive'
PROJECT_FOLDER = DRIVE_MOUNT + '/MyDrive/DataMining/project_dataset'

REAL_DIR = BASE_PATH + '/Real-img'
FAKE_DIR = BASE_PATH + '/Image'

if not os.path.ismount(DRIVE_MOUNT):
    drive.mount(DRIVE_MOUNT)

def quick_extract(zip_name, target_path):
    if not os.path.exists(target_path):
        zip_path = os.path.join(PROJECT_FOLDER, zip_name)
        print(f"Extracting {zip_name} to {target_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(BASE_PATH)
    else:
        print(f"{target_path} already exists.")

quick_extract('Real-img.zip', REAL_DIR)
quick_extract('Fake-img.zip', FAKE_DIR)

## 2. Robust Sampler Logic

In [ ]:
def find_image_robust(directory, img_id):
    """
    Attempts to find an image in the directory matches the ID, 
    trying several common HiDF naming variations.
    """
    exts = ['.jpg', '.png', '.jpeg']
    
    # 1. Try exactly as provided
    for ext in exts:
        path = os.path.join(directory, f"{img_id}{ext}")
        if os.path.exists(path): return path
    
    # 2. Try removing leading 'f' or 'F' if present
    if img_id.lower().startswith('f'):
        stripped_id = img_id[1:]
        for ext in exts:
            path = os.path.join(directory, f"{stripped_id}{ext}")
            if os.path.exists(path): return path
            
    # 3. Try adding leading 'f' if not present
    if not img_id.lower().startswith('f'):
        f_id = f"f{img_id}"
        for ext in exts:
            path = os.path.join(directory, f"{f_id}{ext}")
            if os.path.exists(path): return path
            
    # 4. Try Case Insensitive Search (Slowest, but guaranteed if it exists)
    all_files = os.listdir(directory)
    search_id = img_id.lower()
    for f in all_files:
        if f.lower().startswith(search_id) or (search_id.startswith('f') and f.lower().startswith(search_id[1:])):
            return os.path.join(directory, f)
            
    return None

def display_deepfake_ancestry_v2(num_samples=3):
    fake_files = [f for f in os.listdir(FAKE_DIR) if f.endswith('.jpg')]
    samples = random.sample(fake_files, num_samples)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = [axes]

    for i, fake_file in enumerate(samples):
        parts = fake_file.split('.')[0].split('_')
        if len(parts) < 2: continue
        base_id, swap_id = parts[0], parts[1]
        
        # Load Results
        img_fake = Image.open(os.path.join(FAKE_DIR, fake_file))
        
        # Find source paths robustly
        base_path = find_image_robust(REAL_DIR, base_id)
        swap_path = find_image_robust(REAL_DIR, swap_id)
        
        # Column 0: Base
        if base_path:
            axes[i][0].imshow(Image.open(base_path))
            axes[i][0].set_title(f"Base: {base_id}")
        else:
            axes[i][0].text(0.5, 0.5, f"MISSING:\n{base_id}", ha='center', va='center')
            axes[i][0].set_title(f"Base ID: {base_id} (Not Found)")
        axes[i][0].axis('off')
        
        # Column 1: Swap
        if swap_path:
            axes[i][1].imshow(Image.open(swap_path))
            axes[i][1].set_title(f"Swap: {swap_id}")
        else:
            axes[i][1].text(0.5, 0.5, f"MISSING:\n{swap_id}", ha='center', va='center')
            axes[i][1].set_title(f"Swap ID: {swap_id} (Not Found)")
        axes[i][1].axis('off')
        
        # Column 2: Result
        axes[i][2].imshow(img_fake)
        axes[i][2].set_title("Generated Deepfake")
        for spine in axes[i][2].spines.values():
            spine.set_edgecolor('red')
            spine.set_linewidth(3)
        axes[i][2].axis('off')

    plt.tight_layout()
    plt.show()

# Run the updated visualization
display_deepfake_ancestry_v2(num_samples=3)